# 03 - Baselines, models, validation and backtest

Run `python -m src.cli prepare --tickers AAPL` first. The test set is touched only after model selection.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src import backtest as bt
from src import evaluate as ev
from src import models as mmod
from src.features import build_feature_list, split_time_series
from src.utils import FIGURES_DIR, ensure_dirs, set_seed

ensure_dirs()
set_seed(42)
df = pd.read_csv(ROOT / 'data/processed/AAPL_features.csv', index_col=0, parse_dates=True)
features = build_feature_list(df)
train, test = split_time_series(df, 0.8)
X_train, y_train = train[features], train['target']
X_test, y_test = test[features], test['target']
print(len(X_train), len(X_test), len(features))

## Baselines

In [ ]:
ev.baseline_metrics(test)

## Walk-forward CV on the training period only

In [ ]:
model = mmod.build_model('xgboost')
scores, summary = mmod.walk_forward_scores(model, X_train, y_train, n_splits=5)
print(scores.to_string(index=False))
print(summary)

## Final model - evaluate the untouched test set once

In [ ]:
model.fit(X_train, y_train)
probs = mmod.predict_proba({'model': model, 'features': features}, X_test)
metrics = mmod.evaluate_predictions(y_test, probs)
print({k: round(v, 4) for k, v in metrics.items() if isinstance(v, float)})
print('Confident subset:', mmod.confident_subset_metrics(y_test, probs))

## Cost-aware backtest

In [ ]:
result = bt.backtest(test, probs, threshold=0.55, cost=0.001)
print(bt.backtest_table(result).to_string(index=False))
bt.plot_equity_curves({'AAPL xgboost': result}, FIGURES_DIR / 'notebook_equity.png')
plt.show()